# Fine-Tuning LFM3-1.7B for Structured Extraction with TRL (SFT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/lfm3_sft_with_trl.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-Liquid4All%2Fcookbook-181717?logo=github)](https://github.com/Liquid4All/cookbook) [![Discord](https://img.shields.io/discord/1385439864920739850?color=7289da&label=Discord&logo=discord&logoColor=white)](https://discord.com/invite/liquid-ai)

This notebook fine-tunes [LFM3-1.7B](https://huggingface.co/LiquidAI/LFM3-1.7B) to turn messy real-world documents (emails, receipts, booking confirmations, forms in many languages) into clean JSON that follows a schema. We use Supervised Fine-Tuning (SFT) with [TRL](https://github.com/huggingface/trl) and LoRA, and measure field-level accuracy before and after training, including on document types the model never saw during training.

Extraction is a typical on-device task: the documents stay on the device, and a small model that reliably fills your schema is faster and cheaper than a large general one.

## 🎯 What you'll find

- **The task**: a JSON schema and a document go in, a JSON object comes out
- **LoRA + SFT**: train on the JSON answers only, in LFM3's non-reasoning mode
- **Evaluation**: valid JSON rate, field accuracy, and exact match, on seen and unseen document types

# 📦 Installation & Setup

We pin every library to the versions this notebook was tested with.

In [ ]:
!pip install -qqq "torch==2.14.1" "torchvision==0.29.1" "torchao==0.18.0" "trl==1.16.0" "transformers==5.20.2" "peft==0.21.2" "accelerate==1.15.0" "datasets==5.1.0" "huggingface-hub==2.3.0" "tokenizers==0.23.3" "safetensors==0.8.0" --progress-bar off

In [ ]:
import json
import os
import torch
import transformers
import trl
import peft

os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")
print(f"🎛️ PEFT version: {peft.__version__}")

# LFM3-1.7B is released in bfloat16. GPUs without bfloat16 support, like the T4, load it in float16 instead:
# the conversion is exact for almost every weight, and the LoRA adapters train in float32 either way.
use_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"🔢 Precision: {dtype}")

# Loading the model from Transformers 🤗

LFM3-1.7B ships its modeling code with the checkpoint, so we load it with `trust_remote_code=True`. We pad on the left, so that batched generation continues each prompt right where it ends.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "LiquidAI/LFM3-1.7B"

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, padding_side="left")

print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, device_map="auto", trust_remote_code=True)

print("✅ Model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")

# 📚 Load the extraction dataset

Each example has a document type, its JSON schema, a document, and the gold extraction. The documents are fictional but realistic, in 16 languages, and some leave fields unanswered on purpose: the gold value is then `null`, and the model must not invent one.

The dataset has three splits: `train`, `test` (new documents of the training document types), and `test_unseen` (document types that never appear in training).

In [ ]:
from datasets import load_dataset

dataset = load_dataset("LiquidAI/structured-extraction")
print(dataset)

example = dataset["train"][0]
print(f"\n📄 Document type: {example['doc_type']} ({example['language']})")
print(f"\n📝 Document:\n{example['document'][:800]}")
print(f"\n🎯 Gold extraction:\n{json.dumps(json.loads(example['extraction']), indent=2, ensure_ascii=False)[:800]}")

## Build the prompts

The system prompt holds the extraction rules the dataset follows (date and number formats, `null` for missing information, ...) and the schema. The document goes in the user message, and the answer is the JSON object, on one line. Both the original and the fine-tuned model get the same instructions.

Extraction doesn't need step-by-step reasoning, so we use LFM3's non-reasoning mode (`enable_thinking=False`): the answer comes right away, which is faster and cheaper. TRL reads the `chat_template_kwargs` column and passes it to the chat template.

In [ ]:
SYSTEM_PROMPT = """Extract the information from the document into a JSON object that follows the schema below, with these rules:

1. Output exactly one JSON object with exactly the keys of the schema, in schema order, at every nesting level. No extra keys, no prose, no code fences.
2. String values are copied from the document exactly as written: same language, script, spelling and capitalization. Never translate or paraphrase. Leave out field labels ("Order no.:"), surrounding quotes and trailing punctuation. For person names, leave out honorifics and courtesy titles (Mr, Mrs, Dr, Prof, Herr, Frau, Mme, M., Sr., Sra., 様, さん, 先生, 님, г-н). Phone numbers, IDs, emails, URLs and codes are copied as written. A value printed over several lines is joined with ", ".
3. Dates are `YYYY-MM-DD` and times are `HH:MM` on a 24-hour clock, whatever the document's local format, language or calendar (a Buddhist-era year 2569 is 2026).
4. Amounts and quantities are JSON numbers: no currency symbol, no thousands separator, dot as the decimal separator (`1.234,50 €` -> `1234.5`, `¥12,800` -> `12800`).
5. A currency field holds the ISO 4217 code (EUR, USD, JPY, ...) of the amounts in the document.
6. Booleans are `true` or `false` only when the document states it; otherwise `null`.
7. Enum fields take one of the schema's allowed values, chosen by meaning even when the document is in another language.
8. If the document does not contain the information for a field, the value is `null`. Never guess, infer from outside knowledge, or compute a value that is not printed. An array with no entries is `[]`.
9. Array items follow the order they appear in the document.
10. When the document mentions several candidate values (an old and a new date, a previous balance and the amount due, a CC'd person and an attendee), use the one the field description asks for; updated values replace superseded ones.

Schema:
{schema}"""

def build_messages(example):
    return [
        {"role": "system", "content": SYSTEM_PROMPT.format(schema=example["schema"])},
        {"role": "user", "content": example["document"]},
    ]

def to_prompt_completion(example):
    answer = json.dumps(json.loads(example["extraction"]), ensure_ascii=False)
    return {
        "prompt": build_messages(example),
        "completion": [{"role": "assistant", "content": answer}],
        "chat_template_kwargs": {"enable_thinking": False},
    }

train_dataset = dataset["train"].map(to_prompt_completion, remove_columns=dataset["train"].column_names)
print(tokenizer.apply_chat_template(train_dataset[0]["prompt"] + train_dataset[0]["completion"], tokenize=False, enable_thinking=False)[-700:])

# 📏 Evaluate the model before training

We generate an extraction for each evaluation document, with greedy decoding and the repetition penalty recommended in the model card (`repetition_penalty=1.1`), and compare it with the gold JSON:

- **Valid JSON**: the output parses as a JSON object (we tolerate a Markdown code fence around it, even though the rules ask for none).
- **Field accuracy**: the share of fields with the right value. Strings are compared without case and extra spaces, numbers as numbers, and `null` must stay `null`.
- **Exact match**: every field is right.

To keep the notebook fast, we evaluate on 200 documents of each test split.

In [ ]:
from tqdm.auto import tqdm

def parse_json(text):
    """The JSON object in a model's answer, tolerating a Markdown code fence around it, or None."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1].rsplit("```", 1)[0]
    try:
        value = json.loads(text)
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None

def flatten(value, path=""):
    """Leaves of a JSON value as {path: value}, e.g. {"items[0].price": 4.5}."""
    if isinstance(value, dict):
        return {k: v for key, child in value.items() for k, v in flatten(child, f"{path}.{key}" if path else key).items()}
    if isinstance(value, list):
        return {k: v for i, child in enumerate(value) for k, v in flatten(child, f"{path}[{i}]").items()} or {path: []}
    return {path: value}

def same(a, b):
    if isinstance(a, bool) or isinstance(b, bool):
        return a is b
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return abs(a - b) < 1e-6
    if isinstance(a, str) and isinstance(b, str):
        return " ".join(a.split()).casefold() == " ".join(b.split()).casefold()
    return a == b

def score(prediction, gold):
    """Field accuracy of one prediction: matching fields / fields in the gold or the prediction."""
    gold_fields, predicted_fields = flatten(gold), flatten(prediction)
    paths = set(gold_fields) | set(predicted_fields)
    if not paths:
        return 1.0
    return sum(p in gold_fields and p in predicted_fields and same(gold_fields[p], predicted_fields[p]) for p in paths) / len(paths)

@torch.no_grad()
def evaluate(model, split, n=200, batch_size=16):
    model.eval()
    examples = dataset[split].shuffle(seed=42).select(range(n))
    valid, accuracy, exact = 0, 0.0, 0
    for start in tqdm(range(0, n, batch_size), desc=f"Evaluating {split}"):
        batch = examples.select(range(start, min(start + batch_size, n)))
        prompts = [tokenizer.apply_chat_template(build_messages(e), add_generation_prompt=True, tokenize=False, enable_thinking=False) for e in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=1024, do_sample=False, repetition_penalty=1.1)
        for output, example in zip(outputs[:, inputs["input_ids"].shape[1]:], batch):
            gold = json.loads(example["extraction"])
            prediction = parse_json(tokenizer.decode(output, skip_special_tokens=True))
            if prediction is None:
                continue  # invalid JSON scores 0 on every metric
            valid += 1
            field_accuracy = score(prediction, gold)
            accuracy += field_accuracy
            exact += field_accuracy == 1.0
    return {"valid_json": valid / n, "field_accuracy": accuracy / n, "exact_match": exact / n}

def show(results, title):
    print(title)
    for split, r in results.items():
        print(f"   {split:<12} valid JSON {r['valid_json']:.1%} | field accuracy {r['field_accuracy']:.1%} | exact match {r['exact_match']:.1%}")

baseline = {split: evaluate(model, split) for split in ["test", "test_unseen"]}
show(baseline, "📏 Before training")

# 🎯 Supervised Fine-Tuning (SFT + LoRA)

## Wrap the model with PEFT

We adapt the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`), the gated short convolutions (`in_proj`, `out_proj`), and the MLP (`w1`, `w2`, `w3`). LFM3's attention layers also have a per-head output gate, `gate_proj`; adding it to the targets is a reasonable choice.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "out_proj", "w1", "w2", "w3",
        # "gate_proj",  # optional: LFM3's per-head attention gate
    ],
    task_type="CAUSAL_LM",
)

## Launch training

With a prompt-completion dataset, `SFTTrainer` computes the loss on the completion only: the model learns to write the JSON, not to reproduce the schema or the document.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="./LFM3-1.7B-extraction",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # <- effective batch size of 16 documents
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,  # <- a float is a fraction of the total steps
    max_length=4096,
    logging_steps=10,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,  # <- float16 mixed precision with loss scaling on GPUs without bfloat16
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

print("🚀 Starting SFT training...")
trainer.train()
print("🎉 SFT training completed!")
print(f"🧮 Peak GPU memory: {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")

# 📊 Evaluate the fine-tuned model

In [ ]:
finetuned = {split: evaluate(trainer.model, split) for split in ["test", "test_unseen"]}
show(baseline, "📏 Before training")
show(finetuned, "📏 After training")

# 🔍 Original vs fine-tuned model

To see what changed, we run the original model and the fine-tuned model on the same documents. With LoRA, we don't need to load the original model again: `disable_adapter()` temporarily turns the adapter off.

The `showcase` split holds a few documents of types the model never saw during training. For each field, we show the gold value and both extractions, with ✅ when the value is right.

In [ ]:
def extract(model, example):
    prompt = tokenizer.apply_chat_template(build_messages(example), add_generation_prompt=True, tokenize=False, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    output = model.generate(**inputs, max_new_tokens=1024, do_sample=False, repetition_penalty=1.1)
    return parse_json(tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))

def cell(fields, path, gold_fields):
    if fields is None:
        return "❌ invalid JSON"
    if path not in fields:
        return "❌ (missing)"
    mark = "✅" if path in gold_fields and same(fields[path], gold_fields[path]) else "❌"
    return f"{mark} {json.dumps(fields[path], ensure_ascii=False)}"

trainer.model.eval()
for example in dataset["showcase"]:
    with trainer.model.disable_adapter():
        original = extract(trainer.model, example)
    finetuned_extraction = extract(trainer.model, example)
    gold_fields = flatten(json.loads(example["extraction"]))
    original_fields = flatten(original) if isinstance(original, dict) else None
    finetuned_fields = flatten(finetuned_extraction) if isinstance(finetuned_extraction, dict) else None
    print(f"\n📄 {example['doc_type']} ({example['language']})\n{example['document'][:400]}...\n")
    print(f"{'field':<28} {'gold':<30} {'original':<34} {'fine-tuned'}")
    for path, gold_value in gold_fields.items():
        print(f"{path[:27]:<28} {json.dumps(gold_value, ensure_ascii=False)[:29]:<30} {cell(original_fields, path, gold_fields)[:33]:<34} {cell(finetuned_fields, path, gold_fields)[:40]}")

And the summary over all the evaluation documents:

In [ ]:
print(f"{'':<34}{'original':>10}{'fine-tuned':>12}")
for split, name in [("test", "seen document types"), ("test_unseen", "unseen document types")]:
    for metric in ["valid_json", "field_accuracy", "exact_match"]:
        print(f"{name + ', ' + metric:<34}{baseline[split][metric]:>10.1%}{finetuned[split][metric]:>12.1%}")

# 💾 Save and merge the model

We save the LoRA adapter, then merge it into the base weights. `save_pretrained` also copies LFM3's modeling code next to the weights, so the merged model loads with `trust_remote_code=True` like the original one.

In [ ]:
trainer.save_model("./LFM3-1.7B-extraction-lora")

output_dir = "./LFM3-1.7B-extraction-merged"
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"💾 Merged model saved to: {output_dir}")

# To push the merged model to the Hugging Face Hub, uncomment:
# merged_model.push_to_hub("your-username/LFM3-1.7B-extraction")
# tokenizer.push_to_hub("your-username/LFM3-1.7B-extraction")

# 🚀 Next steps

- **Your own schemas**: write a JSON schema per document type and a few hundred labeled documents. Include documents with missing fields, so the model learns to return `null` instead of guessing.
- **Constrained decoding**: at inference, a JSON-schema grammar (supported by llama.cpp and vLLM) guarantees valid JSON on top of the fine-tuned accuracy.